# NB17 — STRICT_6 Redundancy Sensitivity and Descriptive Morphometry

This notebook adds a strict six-descriptor representation by removing `EquivDiameter` from REDUCED_7 because `EquivDiameter = sqrt(4*Area/pi)`. It also generates cultivar-wise descriptive statistics in the original analytical units. No conversion to millimeters is performed because the analytical dataset does not contain a validated pixel-to-physical calibration.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install "tabpfn==9.0.0" xgboost scipy scikit-learn pandas openpyxl statsmodels
from pathlib import Path
from getpass import getpass
import os, json, warnings, gc
import numpy as np, pandas as pd
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, log_loss
from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier
warnings.filterwarnings('ignore')
ROOT=Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1'); DATASET=ROOT/'01_DATA'/'INIAP_Dataset.xlsx'; OUT=ROOT/'03_RESULTS'/'NB17_STRICT6'; TAB=ROOT/'06_TABLES'/'NB17_STRICT6'; CKPT=OUT/'checkpoints'
for p in [OUT,TAB,CKPT]: p.mkdir(parents=True,exist_ok=True)
token=os.environ.get('TABPFN_TOKEN','').strip()
if not token:
    try: token=str(userdata.get('TABPFN_TOKEN') or '').strip()
    except Exception: token=''
if not token: token=getpass('Prior Labs API key (hidden): ').strip()
if token: os.environ['TABPFN_TOKEN']=token
df=pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
F11=['Area','Perimeter','MajorAxisLength','MinorAxisLength','AspectRatio','ConvexArea','EquivDiameter','Extent','Solidity','roundness','Compactness']; F7=[c for c in F11 if c not in ['AspectRatio','Solidity','roundness','Compactness']]; F6=[c for c in F7 if c!='EquivDiameter']
le=LabelEncoder(); y=pd.Series(le.fit_transform(df['Class'].astype(str)),index=df.index); CLASSES=list(le.classes_); SEEDS=[2026,2027,2028]


In [ ]:
identity=pd.DataFrame([
['AspectRatio = MajorAxisLength / MinorAxisLength',np.max(np.abs(df.AspectRatio-df.MajorAxisLength/df.MinorAxisLength))],
['Solidity = Area / ConvexArea',np.max(np.abs(df.Solidity-df.Area/df.ConvexArea))],
['roundness = 4*pi*Area / Perimeter^2',np.max(np.abs(df.roundness-4*np.pi*df.Area/(df.Perimeter**2)))],
['Compactness = EquivDiameter / MajorAxisLength',np.max(np.abs(df.Compactness-df.EquivDiameter/df.MajorAxisLength))],
['EquivDiameter = sqrt(4*Area/pi)',np.max(np.abs(df.EquivDiameter-np.sqrt(4*df.Area/np.pi)))]],columns=['relation','max_abs_error']); identity.to_csv(TAB/'deterministic_identity_audit_strict6.csv',index=False)
rows=[]
for cls in CLASSES:
    g=df[df.Class.astype(str)==cls]
    for f in F11:
        unit='pixel²' if f in ['Area','ConvexArea'] else ('pixels' if f in ['Perimeter','MajorAxisLength','MinorAxisLength','EquivDiameter'] else 'dimensionless')
        rows.append([cls,f,unit,len(g),g[f].mean(),g[f].std(ddof=1),g[f].median(),g[f].quantile(.25),g[f].quantile(.75)])
pd.DataFrame(rows,columns=['cultivar','feature','unit','n','mean','sd','median','q1','q3']).to_csv(TAB/'morphometrics_by_cultivar_descriptive_original_units.csv',index=False)
kw=[]
for f in F11:
    groups=[df.loc[df.Class.astype(str)==c,f].to_numpy() for c in CLASSES]; H,p=kruskal(*groups); k=len(groups); n=sum(map(len,groups)); eps=max(0,min(1,(H-k+1)/(n-k))); kw.append([f,H,p,eps])
kw=pd.DataFrame(kw,columns=['feature','kruskal_H','p_raw','epsilon_squared']); kw['p_holm']=multipletests(kw.p_raw,method='holm')[1]; kw.to_csv(TAB/'kruskal_holm_epsilon_squared.csv',index=False)


In [ ]:
def registry(seed):
    return {
'LogisticRegression':(Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(max_iter=5000,random_state=seed))]),{'clf__C':[.1,1,10]}),
'SVM_RBF':(Pipeline([('scale',StandardScaler()),('clf',SVC(kernel='rbf',probability=True,random_state=seed))]),{'clf__C':[1,10,100],'clf__gamma':['scale',.01,.1]}),
'XGBoost':(XGBClassifier(objective='multi:softprob',eval_metric='mlogloss',tree_method='hist',n_jobs=1,random_state=seed),{'n_estimators':[300,600],'max_depth':[3,5],'learning_rate':[.03,.1],'subsample':[.8,1],'colsample_bytree':[.8,1]}),
'LDA':(Pipeline([('scale',StandardScaler()),('clf',LinearDiscriminantAnalysis())]),None),
'QDA':(Pipeline([('scale',StandardScaler()),('clf',QuadraticDiscriminantAnalysis())]),{'clf__reg_param':[.01,.05,.1,.2,.5]}),
'KNN':(Pipeline([('scale',StandardScaler()),('clf',KNeighborsClassifier())]),{'clf__n_neighbors':[3,5,7,11,15],'clf__weights':['uniform','distance'],'clf__p':[1,2]}),
'LDA_XGBoost':(Pipeline([('scale',StandardScaler()),('dr',LinearDiscriminantAnalysis()),('clf',XGBClassifier(objective='multi:softprob',eval_metric='mlogloss',tree_method='hist',n_jobs=1,random_state=seed))]),{'dr__n_components':[1,2,3],'clf__n_estimators':[300,600],'clf__max_depth':[3,5],'clf__learning_rate':[.03,.1]})}
MODELS=['LogisticRegression','SVM_RBF','XGBoost','LDA','QDA','KNN','LDA_XGBoost','TabPFN']
def evaluate(model,X,seed,fold,tr,te):
    Xtr,Xte=X.iloc[tr],X.iloc[te]; ytr,yte=y.iloc[tr],y.iloc[te]
    if model=='TabPFN':
        m=TabPFNClassifier(device='cuda'); m.fit(Xtr.to_numpy(np.float32),ytr.to_numpy()); P=m.predict_proba(Xte.to_numpy(np.float32)); pred=P.argmax(1); params={}; del m; gc.collect()
    else:
        est,grid=registry(seed+fold)[model]
        if grid:
            cv=StratifiedKFold(3,shuffle=True,random_state=seed+fold); gs=GridSearchCV(est,grid,scoring='f1_macro',cv=cv,n_jobs=-1,refit=True); gs.fit(Xtr,ytr); m=gs.best_estimator_; params=gs.best_params_
        else: m=est.fit(Xtr,ytr); params={}
        P=m.predict_proba(Xte) if hasattr(m,'predict_proba') else None; pred=m.predict(Xte)
    return {'f1_macro':f1_score(yte,pred,average='macro'),'accuracy':accuracy_score(yte,pred),'log_loss':log_loss(yte,P,labels=np.arange(len(CLASSES))) if P is not None else np.nan,'params':json.dumps(params)}


In [ ]:
X6=df[F6].copy(); rows=[]
for seed in SEEDS:
    outer=StratifiedKFold(5,shuffle=True,random_state=seed)
    for fold,(tr,te) in enumerate(outer.split(X6,y),1):
        for model in MODELS:
            cp=CKPT/f'STRICT6_{model}_seed{seed}_fold{fold}.csv'
            if cp.exists(): r=pd.read_csv(cp).iloc[0].to_dict()
            else:
                r=evaluate(model,X6,seed,fold,tr,te); r.update({'feature_set':'STRICT_6','model':model,'seed':seed,'outer_fold':fold}); pd.DataFrame([r]).to_csv(cp,index=False)
            rows.append(r)
metrics=pd.DataFrame(rows).drop_duplicates(['feature_set','model','seed','outer_fold']); metrics.to_csv(OUT/'strict6_metrics_by_fold.csv',index=False)
summary=metrics.groupby('model',as_index=False).agg(mean_macro_f1=('f1_macro','mean'),sd_macro_f1=('f1_macro','std'),mean_accuracy=('accuracy','mean'),mean_log_loss=('log_loss','mean')); summary.to_csv(TAB/'strict6_model_summary.csv',index=False); display(summary.sort_values('mean_macro_f1',ascending=False))
